# Nhiệm vụ 8: Phân tích Hồ sơ Cụm (Cluster Profiling) — K-Means Baseline

**Mục tiêu nhiệm vụ:** Khắc họa và nhận diện rõ nét tính chất kinh tế của 2 cụm cổ phiếu (K=2) dựa trên thuật toán K-Means Baseline qua lăng kính 8 đặc trưng cốt lõi (Unscaled Features):
- **Phần 8.1:** Tổng hợp feature theo cụm
- **Phần 8.2:** So sánh giữa các cụm
- **Phần 8.3:** Giới hạn diễn giải

**Quy trình thực hiện:**
1. Đọc hồ sơ 30 profile từ `M2/artifacts/m2-evaluation-kmeans/cluster_profiles.csv` và danh sách thành viên từ `cluster_profile_members.csv`.
2. Tính trung bình 8 feature gốc của các thành viên trong từng cụm xuyên suốt 15 tháng development (8.1).
3. So sánh đối đầu 8 feature để xác định động lực phân tách (Thanh khoản, Beta, Momentum, Rủi ro) qua 15 tháng và theo dõi chuỗi quy mô & thanh khoản (8.2).
4. Thiết lập ranh giới diễn giải học thuật theo đúng nguyên tắc bất biến của DELTA (8.3).


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

eval_dir = '../artifacts/m2-evaluation-kmeans' if os.path.exists('../artifacts/m2-evaluation-kmeans') else 'M2/artifacts/m2-evaluation-kmeans'
df_profiles = pd.read_csv(os.path.join(eval_dir, 'cluster_profiles.csv'))
df_members = pd.read_csv(os.path.join(eval_dir, 'cluster_profile_members.csv'))

print(f"Tổng số bản ghi hồ sơ cụm: {len(df_profiles)} (15 tháng x 2 cụm = 30 profile)")


Tổng số bản ghi hồ sơ cụm: 30 (15 tháng x 2 cụm = 30 profile)


### Phần 8.1 — Tổng hợp feature theo cụm

**Mục đích:** Xác định đặc điểm điển hình bằng cách tính trung bình 8 feature gốc của các thành viên trong từng cụm cùng size và nhãn qua 15 tháng development.


In [2]:
feat_cols = ['size', 'liquidity_21_mean', 'beta_126_mean', 'vol_63_mean', 'mdd_126_mean', 'mom_21_mean', 'mom_63_mean', 'mom_126_mean', 'mom_252_mean']
summary_profiles = df_profiles.groupby('aligned_cluster_id')[feat_cols].mean()
summary_profiles.index = ['Cụm 0', 'Cụm 1']
print("=== HỒ SƠ TRUNG BÌNH 8 ĐẶC TRƯNG GỐC QUA 15 THÁNG ===")
print(summary_profiles.round(4).to_string())


=== HỒ SƠ TRUNG BÌNH 8 ĐẶC TRƯNG GỐC QUA 15 THÁNG ===
           size  liquidity_21_mean  beta_126_mean  vol_63_mean  mdd_126_mean  mom_21_mean  mom_63_mean  mom_126_mean  mom_252_mean
Cụm 0   14.7333       3.718099e+11         1.3013       0.2972       -0.1928       0.0265       0.0785        0.1335        0.3938
Cụm 1  359.6000       1.494419e+10         0.6080       0.3537       -0.2109       0.0138       0.0290        0.0555        0.1877


### Phần 8.2 — So sánh giữa các cụm

**Mục đích:** Xác định feature nào làm các cụm khác nhau; so sánh động lượng, rủi ro, beta và thanh khoản giữa các cụm.


In [3]:
p0_m = summary_profiles.loc['Cụm 0']
p1_m = summary_profiles.loc['Cụm 1']

df_compare = pd.DataFrame({
    'Đặc trưng': ['size', 'liquidity_21 (tỷ VND)', 'beta_126', 'vol_63', 'mdd_126', 'mom_21', 'mom_63', 'mom_126', 'mom_252'],
    'Cụm 0 (mean 15 tháng)': [p0_m['size'], p0_m['liquidity_21_mean']/1e9, p0_m['beta_126_mean'], p0_m['vol_63_mean'], p0_m['mdd_126_mean'], p0_m['mom_21_mean'], p0_m['mom_63_mean'], p0_m['mom_126_mean'], p0_m['mom_252_mean']],
    'Cụm 1 (mean 15 tháng)': [p1_m['size'], p1_m['liquidity_21_mean']/1e9, p1_m['beta_126_mean'], p1_m['vol_63_mean'], p1_m['mdd_126_mean'], p1_m['mom_21_mean'], p1_m['mom_63_mean'], p1_m['mom_126_mean'], p1_m['mom_252_mean']],
})
print("=== SO SÁNH PHÂN TÁCH GIỮA CỤM 0 VÀ CỤM 1 QUA 15 THÁNG ===")
print(df_compare.round(4).to_string(index=False))


=== SO SÁNH PHÂN TÁCH GIỮA CỤM 0 VÀ CỤM 1 QUA 15 THÁNG ===
            Đặc trưng  Cụm 0 (mean 15 tháng)  Cụm 1 (mean 15 tháng)
                 size                14.7333               359.6000
liquidity_21 (tỷ VND)               371.8099                14.9442
             beta_126                 1.3013                 0.6080
               vol_63                 0.2972                 0.3537
              mdd_126                -0.1928                -0.2109
               mom_21                 0.0265                 0.0138
               mom_63                 0.0785                 0.0290
              mom_126                 0.1335                 0.0555
              mom_252                 0.3938                 0.1877


In [4]:
p0_all = df_profiles[df_profiles['aligned_cluster_id'] == 0].set_index('snapshot_date')
p1_all = df_profiles[df_profiles['aligned_cluster_id'] == 1].set_index('snapshot_date')
df_dev_table = pd.DataFrame({
    'Snapshot': p0_all.index,
    'Size cụm 0': p0_all['size'].values,
    'Size cụm 1': p1_all['size'].values,
    'Liquidity cụm 0 — tỷ VND/phiên': p0_all['liquidity_21_mean'].values / 1e9,
    'Liquidity cụm 1 — tỷ VND/phiên': p1_all['liquidity_21_mean'].values / 1e9,
})
print("=== QUY MÔ VÀ MEAN LIQUIDITY QUA TOÀN DEVELOPMENT (15 SNAPSHOTS) ===")
print(df_dev_table.round(6).to_string(index=False))


=== QUY MÔ VÀ MEAN LIQUIDITY QUA TOÀN DEVELOPMENT (15 SNAPSHOTS) ===
  Snapshot  Size cụm 0  Size cụm 1  Liquidity cụm 0 — tỷ VND/phiên  Liquidity cụm 1 — tỷ VND/phiên
2023-11-30           9         133                      201.227713                        8.895114
2023-12-29          16         179                      332.434746                       14.506566
2024-01-31          17         179                      295.174249                       14.266154
2024-02-29          17         179                      359.151771                       16.682277
2024-03-29          15         182                      492.019377                       26.096221
2024-04-26          14         200                      475.605616                       23.666850
2024-05-31          15         206                      419.718682                       24.656235
2024-06-28          11         229                      576.264143                       29.329016
2024-07-31          17         230      

### Phần 8.3 — Giới hạn diễn giải

**Mục đích:** Tuân thủ các nguyên tắc bất biến trong nghiên cứu:
- Mean nhạy với quan sát cực đoan. Không suy diễn phân phối của từng cổ phiếu chỉ từ giá trị mean của cụm.
- Chênh lệch profile không tự chứng minh nguyên nhân kinh tế hay hiệu quả đầu tư; không kết luận cụm nào nên mua.
- Nhãn trung tính "Cụm 0" và "Cụm 1" được duy trì, không gán nhãn chủ quan như "siêu cổ phiếu", "tinh hoa", hay "penny".
- Ranh giới Milestone M2 và M3: Đánh giá tỷ suất sinh lời hay Sharpe ratio thuộc về bước Backtest (M3).
